# Ion-Atom Phonon Blockade CNOT Simulation\n\nThis notebook reproduces the main numerical simulations in arXiv:2602.19222v2, **Ion-atom two-qubit quantum gate based on phonon blockade**.\n\nWe will do three things:\n\n1. Compute the Rydberg-induced shifted ion trap frequency and equilibrium position.\n2. Simulate the three-pulse atom-ion CNOT protocol in the ion red-sideband subspace.\n3. Scan the CNOT fidelity as a function of the atomic Rabi frequency.\n\nEach simulation section starts with the physical formula being used, then runs concrete Colab code and displays the result.

## Step 1 - Prepare the Colab environment\n\nFirst we fetch the simulation code from GitHub, install the small Python dependency set, and import the module. If you are running this notebook inside the cloned project already, the cell will use the local files instead of cloning again.

In [ ]:
from pathlib import Path\nimport os\nimport subprocess\nimport sys\n\nREPO_URL = 'https://github.com/WindSweeps/Simulations.git'\ncwd = Path.cwd()\n\nif (cwd / 'ion_atom_gate.py').exists():\n    PROJECT_DIR = cwd\nelif (cwd / 'ion_atom_phonon_blockade_cnot' / 'ion_atom_gate.py').exists():\n    PROJECT_DIR = cwd / 'ion_atom_phonon_blockade_cnot'\nelse:\n    repo_dir = Path('/content/Simulations')\n    if not repo_dir.exists():\n        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)\n    PROJECT_DIR = repo_dir / 'ion_atom_phonon_blockade_cnot'\n\nsubprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements-colab.txt')], check=True)\nos.chdir(PROJECT_DIR)\nsys.path.insert(0, str(PROJECT_DIR))\n\nprint('Project directory:', PROJECT_DIR)\nprint('Files:', sorted(p.name for p in PROJECT_DIR.iterdir() if p.is_file()))

## Step 2 - Define the physical parameters\n\nThe paper uses a $^{87}$Rb atom and a $^9$Be$^+$ ion. The default parameters below are chosen to match the paper's numerical example:\n\n- unperturbed ion trap frequency: $\omega_i / 2\pi = 11.2\,\mathrm{MHz}$;\n- ion-atom distance: $x_0 = 2.57\,\mu\mathrm{m}$;\n- Rydberg atom level: $n=90$;\n- Lamb-Dicke parameter: $\eta = 0.1$;\n- ion Rabi frequency: $\Omega_i / 2\pi = 1\,\mathrm{MHz}$;\n- atom Rabi frequency: $\Omega_a / 2\pi = 1\,\mathrm{GHz}$.\n\nThe parameter `rydberg_phonon_leakage_scale` is an effective knob for nonideal phonon leakage during the fast Rydberg pulses. The default value calibrates this compact truncated model to the paper's reported fidelity near 90%.

In [ ]:
import json\nfrom IPython.display import Image, display\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nimport ion_atom_gate as gate\n\nparams = gate.PhysicalParams(\n    ion_atom_distance_um=2.57,\n    omega_atom_ghz=1.0,\n    omega_ion_mhz=1.0,\n    eta_lamb_dicke=0.1,\n    rydberg_phonon_leakage_scale=0.46,\n)\n\nscales = gate.interaction_scales(params)\npd.DataFrame([\n    {'quantity': 'shifted phonon frequency', 'value': scales['omega_bar_rad_s'] / (2*np.pi*1e6), 'unit': 'MHz'},\n    {'quantity': 'frequency shift Delta', 'value': scales['Delta_rad_s'] / (2*np.pi*1e6), 'unit': 'MHz'},\n    {'quantity': 'Rydberg interaction V0', 'value': scales['V0_rad_s'] / (2*np.pi*1e9), 'unit': 'GHz'},\n    {'quantity': 'linear phonon scale U1', 'value': scales['U1_rad_s'] / (2*np.pi*1e9), 'unit': 'GHz'},\n    {'quantity': 'quadratic phonon scale U2', 'value': scales['U2_rad_s'] / (2*np.pi*1e6), 'unit': 'MHz'},\n    {'quantity': 'sideband coupling eta Omega_i', 'value': params.eta_lamb_dicke * params.omega_ion / (2*np.pi*1e6), 'unit': 'MHz'},\n])

## Simulation 1 - Rydberg-induced phonon frequency shift\n\nWhen the atom is excited to the Rydberg state, the long-range ion-atom interaction is approximated by\n\n$$\nV(x_a,x_i) \simeq \frac{C_4}{x_a^4}\left(1 + \frac{4x_i}{x_a} + \frac{4x_i^2}{x_a^2}\right).\n$$\n\nThe quadratic term modifies the ion trap. The shifted trap frequency and equilibrium displacement are\n\n$$\n\bar{\omega}_i^2 = \omega_i^2 - \frac{8C_4}{m_i x_a^6},\qquad\n\bar{x} = -\frac{4C_4}{m_i\bar{\omega}_i^2 x_a^5}.\n$$\n\nThe code below reproduces the Fig. 2 style plot and checks the value at $x_a=2.57\,\mu\mathrm{m}$.

In [ ]:
output_dir = Path('outputs_colab')\noutput_dir.mkdir(exist_ok=True)\n\nshift_df = gate.plot_shifted_trap(params, output_dir / 'fig2_shifted_trap.png')\ndisplay(Image(filename=str(output_dir / 'fig2_shifted_trap.png')))\n\nnearest = shift_df.iloc[(shift_df['distance_um'] - params.ion_atom_distance_um).abs().argmin()]\npd.DataFrame([nearest])

## Simulation 2 - Three-pulse phonon-blockade CNOT dynamics\n\nThe CNOT protocol uses three pulses:\n\n1. An atomic $\pi$ pulse couples $|0\rangle_a \leftrightarrow |r\rangle_a$.\n2. An ion red-sideband $\pi$ pulse couples $|0,n\rangle_i \leftrightarrow |1,n-1\rangle_i$.\n3. A second atomic $\pi$ pulse de-excites $|r\rangle_a \rightarrow |0\rangle_a$.\n\nThe compact Hamiltonians used in the simulation are\n\n$$\nH_a = \frac{\Omega_a}{2}\left(|r\rangle\langle 0| + |0\rangle\langle r|\right)\n+ s_1 U_1 |r\rangle\langle r|(a+a^\dagger)\n+ U_2 |r\rangle\langle r|(a^2+a^{\dagger 2}+aa^\dagger+a^\dagger a),\n$$\n\n$$\nH_i = \frac{\eta\Omega_i}{2}(\sigma_+a+\sigma_-a^\dagger)\n- \Delta |r\rangle\langle r|a^\dagger a + H_{\phi}.\n$$\n\nHere $\Delta=\omega_i-\bar{\omega}_i$. If the atom is in $|r\rangle$, $|\Delta|\gg\eta\Omega_i$ suppresses the red-sideband transition, producing phonon blockade. The extra phase term $H_\phi$ implements the phase choice described in the paper so that the blocked branch returns with the CNOT phase convention.

In [ ]:
trace_00 = gate.segment_trace(params, ('0', '0', params.n_phonon_initial))\ntrace_10 = gate.segment_trace(params, ('1', '0', params.n_phonon_initial))\ntrace_df = pd.concat([trace_00, trace_10], ignore_index=True)\n\ngate.plot_amplitude_traces(trace_df, output_dir / 'fig3_amplitude_traces.png')\ndisplay(Image(filename=str(output_dir / 'fig3_amplitude_traces.png')))\n\ntrace_df.groupby(['initial', 'pulse', 'state'])['probability'].max().reset_index().sort_values(\n    ['initial', 'pulse', 'probability'], ascending=[True, True, False]\n).head(12)

## Simulation 3 - CNOT fidelity versus atomic Rabi frequency\n\nThe logical basis is the atom plus the red-sideband ion-phonon pair:\n\n$$\n\{|0,01\rangle, |0,10\rangle, |1,01\rangle, |1,10\rangle\}.\n$$\n\nAfter the full pulse sequence, the full Hilbert-space unitary is projected onto this logical basis to get an effective operation $M$. We compare it with the ideal CNOT $U_\mathrm{CNOT}$ using the average operation fidelity\n\n$$\nF = \frac{\mathrm{Tr}(M^\dagger M) + |\mathrm{Tr}(U_\mathrm{CNOT}^\dagger M)|^2}{d(d+1)},\qquad d=4.\n$$\n\nThe scan below reproduces the Fig. 4 trend: increasing $\Omega_a$ suppresses unwanted phonon leakage during the Rydberg pulses and improves the gate fidelity.

In [ ]:
omega_values = np.linspace(0.2, 5.0, 49)\nscan_df = gate.scan_fidelity(params, omega_values)\nscan_df.to_csv(output_dir / 'fig4_fidelity_scan.csv', index=False)\n\ngate.plot_fidelity_scan(scan_df, output_dir / 'fig4_fidelity_scan.png')\ndisplay(Image(filename=str(output_dir / 'fig4_fidelity_scan.png')))\n\nat_1ghz = scan_df.iloc[(scan_df['omega_atom_ghz'] - 1.0).abs().argmin()]\npd.DataFrame([at_1ghz, scan_df.iloc[scan_df['fidelity'].idxmax()]])

## Simulation 4 - Run one full parameter set\n\nThis cell runs the complete workflow and writes all figures, CSV files, projected gate matrices, and a JSON summary into `outputs_custom`. Change the numbers at the top of the cell to test a different regime.

In [ ]:
custom_params = gate.PhysicalParams(\n    ion_atom_distance_um=2.57,\n    omega_atom_ghz=1.0,\n    omega_ion_mhz=1.0,\n    eta_lamb_dicke=0.1,\n    rydberg_phonon_leakage_scale=0.46,\n)\n\nsummary = gate.run(custom_params, Path('outputs_custom'))\nprint(json.dumps(summary['scales'], indent=2))\nprint(json.dumps(summary['metrics_at_default'], indent=2))

## Step 3 - Download the generated results\n\nFinally, package the generated figures and data so they can be downloaded from Colab.

In [ ]:
import shutil\n\narchive = shutil.make_archive('ion_atom_phonon_blockade_results', 'zip', root_dir='.', base_dir='outputs_custom')\nprint('Created:', archive)\n\ntry:\n    from google.colab import files\n    files.download(archive)\nexcept Exception:\n    print('Not running inside Colab; download skipped.')